# 🧬 BigQuery Graph Analytics — Drug-Target Interaction Network


## 🧠 The Business Problem

A drug molecule does not just bind to one target. It interacts with multiple proteins, those proteins participate in multiple biological pathways, and those pathways affect multiple disease areas. Understanding the full **blast radius** of a compound is critical for both efficacy and safety.

**Key questions a drug discovery team might ask:**

- What off-target proteins does our lead compound bind to — and what pathways do they affect?
- Which drugs share multiple targets with our compound — potential combo therapy candidates?
- Which compounds avoid a dangerous off-target pathway (e.g. hERG cardiac toxicity)?
- What is the full disease coverage of a compound based on its target-pathway connections?

---

## 🗂️ Dataset Schema

```
bqml-demo-sudipto.drug_target_graph
├── compounds        — drug molecules with mechanism of action
├── targets          — protein targets with gene names
├── interactions     — compound-target binding with affinity data
├── pathways         — biological pathways (KEGG/Reactome style)
└── target_pathways  — which targets participate in which pathways
```

## 🕸️ Graph Model

```
(Compound)-[BINDS_TO {affinity_nm, ic50, interaction_type}]->(Target)
(Target)-[PARTICIPATES_IN {role, importance_score}]->(Pathway)
```

---
## ⚙️ Setup

In [ ]:
!pip install google-cloud-bigquery google-cloud-bigquery[pandas] --quiet

In [1]:
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()

PROJECT  = "bqml-demo-sudipto"
DATASET  = "drug_target_graph"
LOCATION = "US"

client = bigquery.Client(project=PROJECT)
print(f"Connected to project: {PROJECT}")

/usr/local/lib/python3.10/dist-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/dist-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)


Connected to project: bqml-demo-sudipto


In [2]:
%load_ext google.cloud.bigquery

/usr/local/lib/python3.10/dist-packages/google/cloud/bigquery/__init__.py:239: FutureWarning: %load_ext google.cloud.bigquery is deprecated. Install bigquery-magics package and use `%load_ext bigquery_magics`, instead.
  warnings.warn(


---
## 📦 Step 1: Create Dataset

In [3]:
%%bigquery
CREATE SCHEMA IF NOT EXISTS `bqml-demo-sudipto.drug_target_graph`
OPTIONS (location = 'US');

Query is running:   0%|          |

""


---
## 🧪 Step 2: Create `compounds` Table

Drug molecules with mechanism of action, development stage and therapeutic area.

In [4]:
%%bigquery
CREATE OR REPLACE TABLE `bqml-demo-sudipto.drug_target_graph.compounds` AS
SELECT 'CPD001' AS compound_id, 'Imatinib'    AS compound_name, 'Kinase Inhibitor'       AS mechanism_of_action, 'Approved'    AS dev_stage, 'Oncology'           AS therapeutic_area, 479.6 AS molecular_weight UNION ALL
SELECT 'CPD002', 'Gefitinib',   'Kinase Inhibitor',        'Approved',    'Oncology',           446.9 UNION ALL
SELECT 'CPD003', 'Erlotinib',   'Kinase Inhibitor',        'Approved',    'Oncology',           393.4 UNION ALL
SELECT 'CPD004', 'Sorafenib',   'Multi-Kinase Inhibitor',  'Approved',    'Oncology',           464.8 UNION ALL
SELECT 'CPD005', 'Vemurafenib', 'BRAF Inhibitor',          'Approved',    'Oncology',           489.9 UNION ALL
SELECT 'CPD006', 'Crizotinib',  'ALK Inhibitor',           'Approved',    'Oncology',           450.3 UNION ALL
SELECT 'CPD007', 'Idelalisib',  'PI3K Inhibitor',          'Approved',    'Oncology',           415.4 UNION ALL
SELECT 'CPD008', 'Trametinib',  'MEK Inhibitor',           'Approved',    'Oncology',           615.4 UNION ALL
SELECT 'CPD009', 'Palbociclib', 'CDK Inhibitor',           'Approved',    'Oncology',           447.5 UNION ALL
SELECT 'CPD010', 'Compound-X1', 'Kinase Inhibitor',        'Phase II',    'Oncology',           412.3 UNION ALL
SELECT 'CPD011', 'Compound-X2', 'PI3K Inhibitor',          'Phase I',     'Oncology',           398.7 UNION ALL
SELECT 'CPD012', 'Saquinavir',  'Protease Inhibitor',      'Approved',    'Infectious Disease', 670.8 UNION ALL
SELECT 'CPD013', 'Metformin',   'AMPK Activator',          'Approved',    'Metabolic',          165.6 UNION ALL
SELECT 'CPD014', 'Rapamycin',   'mTOR Inhibitor',          'Approved',    'Immunology',         914.2 UNION ALL
SELECT 'CPD015', 'Compound-X3', 'Multi-Kinase Inhibitor',  'Preclinical', 'Oncology',           502.1;

Query is running:   0%|          |

""


---
## 🎯 Step 3: Create `targets` Table

Protein targets with gene names, UniProt IDs and target class.

In [5]:
%%bigquery
CREATE OR REPLACE TABLE `bqml-demo-sudipto.drug_target_graph.targets` AS
SELECT 'TGT001' AS target_id, 'BCR-ABL1'    AS target_name, 'ABL1'    AS gene_name, 'P00519' AS uniprot_id, 'Kinase'       AS target_class, TRUE  AS is_oncogene UNION ALL
SELECT 'TGT002', 'EGFR',      'EGFR',    'P00533', 'Kinase',       TRUE  UNION ALL
SELECT 'TGT003', 'VEGFR2',    'KDR',     'P35968', 'Kinase',       FALSE UNION ALL
SELECT 'TGT004', 'BRAF',      'BRAF',    'P15056', 'Kinase',       TRUE  UNION ALL
SELECT 'TGT005', 'ALK',       'ALK',     'Q9UM73', 'Kinase',       TRUE  UNION ALL
SELECT 'TGT006', 'PI3K-alpha','PIK3CA',  'P42336', 'Lipid Kinase', TRUE  UNION ALL
SELECT 'TGT007', 'MEK1',      'MAP2K1',  'Q02750', 'Kinase',       FALSE UNION ALL
SELECT 'TGT008', 'CDK4',      'CDK4',    'P11802', 'Kinase',       FALSE UNION ALL
SELECT 'TGT009', 'CDK6',      'CDK6',    'P30279', 'Kinase',       FALSE UNION ALL
SELECT 'TGT010', 'mTOR',      'MTOR',    'P42345', 'Kinase',       FALSE UNION ALL
SELECT 'TGT011', 'PDGFR-beta','PDGFRB',  'P09619', 'Kinase',       FALSE UNION ALL
SELECT 'TGT012', 'c-KIT',     'KIT',     'P10721', 'Kinase',       TRUE  UNION ALL
SELECT 'TGT013', 'hERG',      'KCNH2',   'Q12809', 'Ion Channel',  FALSE UNION ALL
SELECT 'TGT014', 'AMPK',      'PRKAA1',  'Q13131', 'Kinase',       FALSE UNION ALL
SELECT 'TGT015', 'RAF1',      'RAF1',    'P04049', 'Kinase',       FALSE UNION ALL
SELECT 'TGT016', 'RET',       'RET',     'P07949', 'Kinase',       TRUE  UNION ALL
SELECT 'TGT017', 'FLT3',      'FLT3',    'P36888', 'Kinase',       TRUE  UNION ALL
SELECT 'TGT018', 'AKT1',      'AKT1',    'P31749', 'Kinase',       FALSE UNION ALL
SELECT 'TGT019', 'ERK2',      'MAPK1',   'P28482', 'Kinase',       FALSE UNION ALL
SELECT 'TGT020', 'HIV-Protease','HIV1-PR','Q72547', 'Protease',     FALSE;

Query is running:   0%|          |

""


---
## 🔗 Step 4: Create `interactions` Table

Compound-target binding data. `interaction_type` flags primary targets vs off-targets. hERG (TGT013) off-target hits indicate cardiac liability.

In [6]:
%%bigquery
CREATE OR REPLACE TABLE `bqml-demo-sudipto.drug_target_graph.interactions` AS
SELECT 'INT001' AS interaction_id, 'CPD001' AS compound_id, 'TGT001' AS target_id, 0.025  AS affinity_nm, 0.038  AS ic50_nm, 'primary'    AS interaction_type, 9.8  AS pchembl_value UNION ALL
SELECT 'INT002', 'CPD001', 'TGT011', 0.1,    0.15,   'primary',    8.8  UNION ALL
SELECT 'INT003', 'CPD001', 'TGT012', 0.068,  0.1,    'primary',    9.2  UNION ALL
SELECT 'INT004', 'CPD001', 'TGT016', 0.5,    0.8,    'off-target', 7.4  UNION ALL
SELECT 'INT005', 'CPD001', 'TGT013', 5.2,    8.1,    'off-target', 6.1  UNION ALL
SELECT 'INT006', 'CPD002', 'TGT002', 0.02,   0.033,  'primary',    9.7  UNION ALL
SELECT 'INT007', 'CPD002', 'TGT013', 3.1,    4.8,    'off-target', 6.5  UNION ALL
SELECT 'INT008', 'CPD003', 'TGT002', 0.058,  0.079,  'primary',    9.2  UNION ALL
SELECT 'INT009', 'CPD003', 'TGT013', 8.5,    12.0,   'off-target', 5.9  UNION ALL
SELECT 'INT010', 'CPD004', 'TGT003', 0.09,   0.12,   'primary',    9.1  UNION ALL
SELECT 'INT011', 'CPD004', 'TGT004', 0.038,  0.055,  'primary',    8.9  UNION ALL
SELECT 'INT012', 'CPD004', 'TGT015', 0.22,   0.31,   'primary',    8.5  UNION ALL
SELECT 'INT013', 'CPD004', 'TGT016', 0.58,   0.75,   'primary',    8.1  UNION ALL
SELECT 'INT014', 'CPD004', 'TGT017', 0.33,   0.48,   'primary',    8.4  UNION ALL
SELECT 'INT015', 'CPD004', 'TGT013', 2.8,    4.1,    'off-target', 6.6  UNION ALL
SELECT 'INT016', 'CPD005', 'TGT004', 0.031,  0.044,  'primary',    9.5  UNION ALL
SELECT 'INT017', 'CPD005', 'TGT015', 0.48,   0.65,   'off-target', 7.3  UNION ALL
SELECT 'INT018', 'CPD006', 'TGT005', 0.02,   0.028,  'primary',    9.8  UNION ALL
SELECT 'INT019', 'CPD006', 'TGT003', 0.41,   0.59,   'off-target', 7.4  UNION ALL
SELECT 'INT020', 'CPD007', 'TGT006', 0.019,  0.025,  'primary',    9.8  UNION ALL
SELECT 'INT021', 'CPD007', 'TGT018', 0.55,   0.78,   'off-target', 7.2  UNION ALL
SELECT 'INT022', 'CPD008', 'TGT007', 0.0092, 0.014,  'primary',    10.1 UNION ALL
SELECT 'INT023', 'CPD008', 'TGT019', 0.38,   0.51,   'off-target', 7.4  UNION ALL
SELECT 'INT024', 'CPD009', 'TGT008', 0.011,  0.017,  'primary',    9.9  UNION ALL
SELECT 'INT025', 'CPD009', 'TGT009', 0.015,  0.022,  'primary',    9.8  UNION ALL
SELECT 'INT026', 'CPD010', 'TGT002', 0.041,  0.058,  'primary',    9.4  UNION ALL
SELECT 'INT027', 'CPD010', 'TGT001', 0.38,   0.52,   'off-target', 7.3  UNION ALL
SELECT 'INT028', 'CPD010', 'TGT013', 1.2,    1.8,    'off-target', 7.1  UNION ALL
SELECT 'INT029', 'CPD011', 'TGT006', 0.029,  0.041,  'primary',    9.5  UNION ALL
SELECT 'INT030', 'CPD011', 'TGT010', 0.71,   0.95,   'off-target', 7.0  UNION ALL
SELECT 'INT031', 'CPD012', 'TGT020', 0.39,   0.55,   'primary',    7.3  UNION ALL
SELECT 'INT032', 'CPD013', 'TGT014', 12.0,   18.5,   'primary',    5.7  UNION ALL
SELECT 'INT033', 'CPD014', 'TGT010', 0.0018, 0.0025, 'primary',    11.8 UNION ALL
SELECT 'INT034', 'CPD015', 'TGT004', 0.055,  0.078,  'primary',    9.2  UNION ALL
SELECT 'INT035', 'CPD015', 'TGT003', 0.12,   0.17,   'primary',    8.9  UNION ALL
SELECT 'INT036', 'CPD015', 'TGT016', 0.29,   0.41,   'primary',    8.5  UNION ALL
SELECT 'INT037', 'CPD015', 'TGT013', 0.95,   1.35,   'off-target', 7.2;

Query is running:   0%|          |

""


---
## 🔬 Step 5: Create `pathways` Table

Biological pathways with associated disease areas and cancer relevance scores.

In [7]:
%%bigquery
CREATE OR REPLACE TABLE `bqml-demo-sudipto.drug_target_graph.pathways` AS
SELECT 'PWY001' AS pathway_id, 'MAPK Signaling'           AS pathway_name, 'hsa04010' AS kegg_id, 'Cell Proliferation'     AS biological_process, 'Oncology'           AS disease_area, 'high'   AS cancer_relevance UNION ALL
SELECT 'PWY002', 'PI3K-AKT Signaling',        'hsa04151', 'Cell Survival',         'Oncology',           'high'   UNION ALL
SELECT 'PWY003', 'VEGF Signaling',             'hsa04370', 'Angiogenesis',          'Oncology',           'high'   UNION ALL
SELECT 'PWY004', 'ErbB Signaling',             'hsa04012', 'Cell Growth',           'Oncology',           'high'   UNION ALL
SELECT 'PWY005', 'mTOR Signaling',             'hsa04150', 'Cell Growth',           'Oncology',           'medium' UNION ALL
SELECT 'PWY006', 'Cell Cycle',                 'hsa04110', 'Cell Division',         'Oncology',           'high'   UNION ALL
SELECT 'PWY007', 'Cardiac Muscle Contraction', 'hsa04260', 'Cardiac Function',      'Cardiovascular',     'low'    UNION ALL
SELECT 'PWY008', 'hERG Cardiac Channel',       'hsa04022', 'Cardiac Repolarisation','Cardiovascular',     'low'    UNION ALL
SELECT 'PWY009', 'AMPK Signaling',             'hsa04152', 'Energy Metabolism',     'Metabolic',          'low'    UNION ALL
SELECT 'PWY010', 'ALK Signaling',              'hsa04915', 'Cell Proliferation',    'Oncology',           'high'   UNION ALL
SELECT 'PWY011', 'RAS Signaling',              'hsa04014', 'Cell Proliferation',    'Oncology',           'high'   UNION ALL
SELECT 'PWY012', 'HIV Replication',            'hsa05170', 'Viral Replication',     'Infectious Disease', 'low';

Query is running:   0%|          |

""


---
## 🔗 Step 6: Create `target_pathways` Table

Junction table linking targets to the pathways they participate in.

In [8]:
%%bigquery
CREATE OR REPLACE TABLE `bqml-demo-sudipto.drug_target_graph.target_pathways` AS
SELECT 'TP001' AS tp_id, 'TGT001' AS target_id, 'PWY001' AS pathway_id, 'activator' AS role, 0.95 AS importance_score UNION ALL
SELECT 'TP002', 'TGT001', 'PWY011', 'activator', 0.88 UNION ALL
SELECT 'TP003', 'TGT002', 'PWY004', 'activator', 0.98 UNION ALL
SELECT 'TP004', 'TGT002', 'PWY001', 'activator', 0.82 UNION ALL
SELECT 'TP005', 'TGT002', 'PWY002', 'activator', 0.75 UNION ALL
SELECT 'TP006', 'TGT003', 'PWY003', 'activator', 0.96 UNION ALL
SELECT 'TP007', 'TGT003', 'PWY001', 'activator', 0.71 UNION ALL
SELECT 'TP008', 'TGT004', 'PWY001', 'activator', 0.97 UNION ALL
SELECT 'TP009', 'TGT004', 'PWY011', 'activator', 0.89 UNION ALL
SELECT 'TP010', 'TGT005', 'PWY010', 'activator', 0.99 UNION ALL
SELECT 'TP011', 'TGT005', 'PWY001', 'activator', 0.78 UNION ALL
SELECT 'TP012', 'TGT006', 'PWY002', 'activator', 0.98 UNION ALL
SELECT 'TP013', 'TGT006', 'PWY005', 'activator', 0.85 UNION ALL
SELECT 'TP014', 'TGT007', 'PWY001', 'activator', 0.94 UNION ALL
SELECT 'TP015', 'TGT007', 'PWY011', 'activator', 0.88 UNION ALL
SELECT 'TP016', 'TGT008', 'PWY006', 'activator', 0.95 UNION ALL
SELECT 'TP017', 'TGT009', 'PWY006', 'activator', 0.93 UNION ALL
SELECT 'TP018', 'TGT010', 'PWY005', 'activator', 0.99 UNION ALL
SELECT 'TP019', 'TGT010', 'PWY002', 'activator', 0.91 UNION ALL
SELECT 'TP020', 'TGT011', 'PWY003', 'activator', 0.87 UNION ALL
SELECT 'TP021', 'TGT011', 'PWY001', 'activator', 0.72 UNION ALL
SELECT 'TP022', 'TGT012', 'PWY001', 'activator', 0.83 UNION ALL
SELECT 'TP023', 'TGT012', 'PWY011', 'activator', 0.79 UNION ALL
SELECT 'TP024', 'TGT013', 'PWY008', 'substrate', 0.99 UNION ALL
SELECT 'TP025', 'TGT013', 'PWY007', 'substrate', 0.95 UNION ALL
SELECT 'TP026', 'TGT014', 'PWY009', 'activator', 0.97 UNION ALL
SELECT 'TP027', 'TGT015', 'PWY001', 'activator', 0.91 UNION ALL
SELECT 'TP028', 'TGT015', 'PWY011', 'activator', 0.86 UNION ALL
SELECT 'TP029', 'TGT016', 'PWY001', 'activator', 0.84 UNION ALL
SELECT 'TP030', 'TGT016', 'PWY003', 'activator', 0.77 UNION ALL
SELECT 'TP031', 'TGT017', 'PWY001', 'activator', 0.88 UNION ALL
SELECT 'TP032', 'TGT017', 'PWY011', 'activator', 0.82 UNION ALL
SELECT 'TP033', 'TGT018', 'PWY002', 'activator', 0.96 UNION ALL
SELECT 'TP034', 'TGT018', 'PWY005', 'activator', 0.88 UNION ALL
SELECT 'TP035', 'TGT019', 'PWY001', 'activator', 0.97 UNION ALL
SELECT 'TP036', 'TGT019', 'PWY011', 'activator', 0.91 UNION ALL
SELECT 'TP037', 'TGT020', 'PWY012', 'substrate', 0.99;

Query is running:   0%|          |

""


---
## 🕸️ Step 7: Create the Property Graph

In [9]:
%%bigquery
CREATE OR REPLACE PROPERTY GRAPH `bqml-demo-sudipto.drug_target_graph.drug_target_interaction_graph`

NODE TABLES (

  `bqml-demo-sudipto.drug_target_graph.compounds`
    AS compound_node
    KEY (compound_id)
    LABEL Compound
    PROPERTIES (compound_id, compound_name, mechanism_of_action, dev_stage, therapeutic_area, molecular_weight),

  `bqml-demo-sudipto.drug_target_graph.targets`
    AS target_node
    KEY (target_id)
    LABEL Target
    PROPERTIES (target_id, target_name, gene_name, uniprot_id, target_class, is_oncogene),

  `bqml-demo-sudipto.drug_target_graph.pathways`
    AS pathway_node
    KEY (pathway_id)
    LABEL Pathway
    PROPERTIES (pathway_id, pathway_name, kegg_id, biological_process, disease_area, cancer_relevance)
)

EDGE TABLES (

  `bqml-demo-sudipto.drug_target_graph.interactions`
    AS binds_to_edge
    KEY (interaction_id)
    SOURCE KEY (compound_id) REFERENCES compound_node (compound_id)
    DESTINATION KEY (target_id) REFERENCES target_node (target_id)
    LABEL BINDS_TO
    PROPERTIES (interaction_id, affinity_nm, ic50_nm, interaction_type, pchembl_value),

  `bqml-demo-sudipto.drug_target_graph.target_pathways`
    AS participates_in_edge
    KEY (tp_id)
    SOURCE KEY (target_id) REFERENCES target_node (target_id)
    DESTINATION KEY (pathway_id) REFERENCES pathway_node (pathway_id)
    LABEL PARTICIPATES_IN
    PROPERTIES (tp_id, role, importance_score)
)

Query is running:   0%|          |

""


---
## ✅ Step 8: Verify the Graph

In [10]:
%%bigquery
SELECT * FROM `bqml-demo-sudipto.drug_target_graph.INFORMATION_SCHEMA.PROPERTY_GRAPHS`;

Query is running:   0%|          |

Downloading:   0%|          |

,property_graph_catalog,property_graph_schema,property_graph_name,property_graph_metadata_json,ddl
0,bqml-demo-sudipto,drug_target_graph,drug_target_interaction_graph,"{""creationTime"":""2026-03-11T07:27:01.933597Z"",...",CREATE PROPERTY GRAPH `bqml-demo-sudipto.drug_...


---
# 🔍 GQL Queries — with SQL Comparisons

Each section shows the **business question**, the **GQL** version, the **equivalent SQL** for side-by-side comparison, and a **graph visualisation** where applicable.

---
## Q1: Full target binding profile per compound
*Show every compound, its targets, binding affinity, and whether primary or off-target*

In [11]:
%%bigquery
-- GQL
SELECT
  compound_name,
  dev_stage,
  target_name,
  gene_name,
  target_class,
  interaction_type,
  ROUND(affinity_nm, 3) AS affinity_nm,
  ROUND(ic50_nm, 3)     AS ic50_nm,
  pchembl_value
FROM GRAPH_TABLE(
  `bqml-demo-sudipto.drug_target_graph.drug_target_interaction_graph`
  MATCH (c:Compound)-[b:BINDS_TO]->(t:Target)
  COLUMNS (
    c.compound_name    AS compound_name,
    c.dev_stage        AS dev_stage,
    t.target_name      AS target_name,
    t.gene_name        AS gene_name,
    t.target_class     AS target_class,
    b.interaction_type AS interaction_type,
    b.affinity_nm      AS affinity_nm,
    b.ic50_nm          AS ic50_nm,
    b.pchembl_value    AS pchembl_value
  )
)
ORDER BY compound_name, interaction_type, affinity_nm;

Query is running:   0%|          |

Downloading:   0%|          |

,compound_name,dev_stage,target_name,gene_name,target_class,interaction_type,affinity_nm,ic50_nm,pchembl_value
0,Compound-X1,Phase II,BCR-ABL1,ABL1,Kinase,off-target,0.380,0.520,7.3
1,Compound-X1,Phase II,hERG,KCNH2,Ion Channel,off-target,1.200,1.800,7.1
2,Compound-X1,Phase II,EGFR,EGFR,Kinase,primary,0.041,0.058,9.4
3,Compound-X2,Phase I,mTOR,MTOR,Kinase,off-target,0.710,0.950,7.0
4,Compound-X2,Phase I,PI3K-alpha,PIK3CA,Lipid Kinase,primary,0.029,0.041,9.5
5,Compound-X3,Preclinical,hERG,KCNH2,Ion Channel,off-target,0.950,1.350,7.2
6,Compound-X3,Preclinical,BRAF,BRAF,Kinase,primary,0.055,0.078,9.2
7,Compound-X3,Preclinical,VEGFR2,KDR,Kinase,primary,0.120,0.170,8.9
8,Compound-X3,Preclinical,RET,RET,Kinase,primary,0.290,0.410,8.5
9,Crizotinib,Approved,VEGFR2,KDR,Kinase,off-target,0.410,0.590,7.4


In [12]:
%%bigquery
-- SQL equivalent
SELECT
  c.compound_name,
  c.dev_stage,
  t.target_name,
  t.gene_name,
  t.target_class,
  i.interaction_type,
  ROUND(i.affinity_nm, 3) AS affinity_nm,
  ROUND(i.ic50_nm, 3)     AS ic50_nm,
  i.pchembl_value
FROM `bqml-demo-sudipto.drug_target_graph.interactions` i
JOIN `bqml-demo-sudipto.drug_target_graph.compounds` c ON i.compound_id = c.compound_id
JOIN `bqml-demo-sudipto.drug_target_graph.targets`   t ON i.target_id   = t.target_id
ORDER BY c.compound_name, i.interaction_type, i.affinity_nm;

Query is running:   0%|          |

Downloading:   0%|          |

,compound_name,dev_stage,target_name,gene_name,target_class,interaction_type,affinity_nm,ic50_nm,pchembl_value
0,Compound-X1,Phase II,BCR-ABL1,ABL1,Kinase,off-target,0.380,0.520,7.3
1,Compound-X1,Phase II,hERG,KCNH2,Ion Channel,off-target,1.200,1.800,7.1
2,Compound-X1,Phase II,EGFR,EGFR,Kinase,primary,0.041,0.058,9.4
3,Compound-X2,Phase I,mTOR,MTOR,Kinase,off-target,0.710,0.950,7.0
4,Compound-X2,Phase I,PI3K-alpha,PIK3CA,Lipid Kinase,primary,0.029,0.041,9.5
5,Compound-X3,Preclinical,hERG,KCNH2,Ion Channel,off-target,0.950,1.350,7.2
6,Compound-X3,Preclinical,BRAF,BRAF,Kinase,primary,0.055,0.078,9.2
7,Compound-X3,Preclinical,VEGFR2,KDR,Kinase,primary,0.120,0.170,8.9
8,Compound-X3,Preclinical,RET,RET,Kinase,primary,0.290,0.410,8.5
9,Crizotinib,Approved,VEGFR2,KDR,Kinase,off-target,0.410,0.590,7.4


---
## Q2: Which compounds have hERG off-target hits — and what cardiac pathways are at risk?
*2-hop traversal: compound → hERG target → cardiac pathway. The classic drug safety question.*

In [13]:
%%bigquery
-- GQL: 2-hop traversal filtered on hERG cardiac target
SELECT
  compound_name,
  dev_stage,
  therapeutic_area,
  target_name,
  ROUND(affinity_nm, 3) AS herg_affinity_nm,
  pathway_name,
  disease_area          AS pathway_disease_area
FROM GRAPH_TABLE(
  `bqml-demo-sudipto.drug_target_graph.drug_target_interaction_graph`
  MATCH (c:Compound)-[b:BINDS_TO]->(t:Target)-[p:PARTICIPATES_IN]->(pw:Pathway)
  WHERE t.gene_name        = 'KCNH2'
    AND b.interaction_type = 'off-target'
    AND pw.disease_area    = 'Cardiovascular'
  COLUMNS (
    c.compound_name    AS compound_name,
    c.dev_stage        AS dev_stage,
    c.therapeutic_area AS therapeutic_area,
    t.target_name      AS target_name,
    b.affinity_nm      AS affinity_nm,
    pw.pathway_name    AS pathway_name,
    pw.disease_area    AS disease_area
  )
)
ORDER BY herg_affinity_nm;

Query is running:   0%|          |

Downloading:   0%|          |

,compound_name,dev_stage,therapeutic_area,target_name,herg_affinity_nm,pathway_name,pathway_disease_area
0,Compound-X3,Preclinical,Oncology,hERG,0.95,Cardiac Muscle Contraction,Cardiovascular
1,Compound-X3,Preclinical,Oncology,hERG,0.95,hERG Cardiac Channel,Cardiovascular
2,Compound-X1,Phase II,Oncology,hERG,1.20,Cardiac Muscle Contraction,Cardiovascular
3,Compound-X1,Phase II,Oncology,hERG,1.20,hERG Cardiac Channel,Cardiovascular
4,Sorafenib,Approved,Oncology,hERG,2.80,Cardiac Muscle Contraction,Cardiovascular
5,Sorafenib,Approved,Oncology,hERG,2.80,hERG Cardiac Channel,Cardiovascular
6,Gefitinib,Approved,Oncology,hERG,3.10,Cardiac Muscle Contraction,Cardiovascular
7,Gefitinib,Approved,Oncology,hERG,3.10,hERG Cardiac Channel,Cardiovascular
8,Imatinib,Approved,Oncology,hERG,5.20,Cardiac Muscle Contraction,Cardiovascular
9,Imatinib,Approved,Oncology,hERG,5.20,hERG Cardiac Channel,Cardiovascular


In [14]:
%%bigquery
-- SQL equivalent: 4 JOINs to traverse the same 2-hop path
SELECT
  c.compound_name,
  c.dev_stage,
  c.therapeutic_area,
  t.target_name,
  ROUND(i.affinity_nm, 3) AS herg_affinity_nm,
  pw.pathway_name,
  pw.disease_area         AS pathway_disease_area
FROM `bqml-demo-sudipto.drug_target_graph.interactions`    i
JOIN `bqml-demo-sudipto.drug_target_graph.compounds`       c  ON i.compound_id  = c.compound_id
JOIN `bqml-demo-sudipto.drug_target_graph.targets`         t  ON i.target_id    = t.target_id
JOIN `bqml-demo-sudipto.drug_target_graph.target_pathways` tp ON t.target_id    = tp.target_id
JOIN `bqml-demo-sudipto.drug_target_graph.pathways`        pw ON tp.pathway_id  = pw.pathway_id
WHERE t.gene_name        = 'KCNH2'
  AND i.interaction_type = 'off-target'
  AND pw.disease_area    = 'Cardiovascular'
ORDER BY i.affinity_nm;
-- 4 JOINs to express what GQL does in one MATCH pattern

Query is running:   0%|          |

Downloading:   0%|          |

,compound_name,dev_stage,therapeutic_area,target_name,herg_affinity_nm,pathway_name,pathway_disease_area
0,Compound-X3,Preclinical,Oncology,hERG,0.95,Cardiac Muscle Contraction,Cardiovascular
1,Compound-X3,Preclinical,Oncology,hERG,0.95,hERG Cardiac Channel,Cardiovascular
2,Compound-X1,Phase II,Oncology,hERG,1.20,Cardiac Muscle Contraction,Cardiovascular
3,Compound-X1,Phase II,Oncology,hERG,1.20,hERG Cardiac Channel,Cardiovascular
4,Sorafenib,Approved,Oncology,hERG,2.80,Cardiac Muscle Contraction,Cardiovascular
5,Sorafenib,Approved,Oncology,hERG,2.80,hERG Cardiac Channel,Cardiovascular
6,Gefitinib,Approved,Oncology,hERG,3.10,Cardiac Muscle Contraction,Cardiovascular
7,Gefitinib,Approved,Oncology,hERG,3.10,hERG Cardiac Channel,Cardiovascular
8,Imatinib,Approved,Oncology,hERG,5.20,Cardiac Muscle Contraction,Cardiovascular
9,Imatinib,Approved,Oncology,hERG,5.20,hERG Cardiac Channel,Cardiovascular


In [15]:
%%bigquery --graph display_only
-- Visualise: compounds with hERG cardiac risk
GRAPH `bqml-demo-sudipto.drug_target_graph.drug_target_interaction_graph`
MATCH (c:Compound)-[b:BINDS_TO]->(t:Target)-[p:PARTICIPATES_IN]->(pw:Pathway)
WHERE t.gene_name     = 'KCNH2'
  AND pw.disease_area = 'Cardiovascular'
RETURN
  TO_JSON(c)  AS compound,
  TO_JSON(b)  AS binds_to,
  TO_JSON(t)  AS target,
  TO_JSON(p)  AS participates_in,
  TO_JSON(pw) AS pathway

Query is running:   0%|          |

Downloading:   0%|          |

---
## Q3: Which compounds share targets — potential combination therapy candidates?
*Find compound pairs that converge on the same protein target node*

In [16]:
%%bigquery
-- GQL: bidirectional convergence on a shared target node
SELECT
  compound_a,
  compound_b,
  shared_target,
  gene_name,
  ROUND(affinity_a_nm, 3) AS affinity_a_nm,
  ROUND(affinity_b_nm, 3) AS affinity_b_nm
FROM GRAPH_TABLE(
  `bqml-demo-sudipto.drug_target_graph.drug_target_interaction_graph`
  MATCH (c1:Compound)-[b1:BINDS_TO]->(t:Target)<-[b2:BINDS_TO]-(c2:Compound)
  WHERE c1.compound_id < c2.compound_id
    AND c1.therapeutic_area = 'Oncology'
    AND c2.therapeutic_area = 'Oncology'
  COLUMNS (
    c1.compound_name AS compound_a,
    c2.compound_name AS compound_b,
    t.target_name    AS shared_target,
    t.gene_name      AS gene_name,
    b1.affinity_nm   AS affinity_a_nm,
    b2.affinity_nm   AS affinity_b_nm
  )
)
ORDER BY compound_a, compound_b, affinity_a_nm;

Query is running:   0%|          |

Downloading:   0%|          |

,compound_a,compound_b,shared_target,gene_name,affinity_a_nm,affinity_b_nm
0,Compound-X1,Compound-X3,hERG,KCNH2,1.200,0.950
1,Crizotinib,Compound-X3,VEGFR2,KDR,0.410,0.120
2,Erlotinib,Compound-X1,EGFR,EGFR,0.058,0.041
3,Erlotinib,Compound-X1,hERG,KCNH2,8.500,1.200
4,Erlotinib,Compound-X3,hERG,KCNH2,8.500,0.950
5,Erlotinib,Sorafenib,hERG,KCNH2,8.500,2.800
6,Gefitinib,Compound-X1,EGFR,EGFR,0.020,0.041
7,Gefitinib,Compound-X1,hERG,KCNH2,3.100,1.200
8,Gefitinib,Compound-X3,hERG,KCNH2,3.100,0.950
9,Gefitinib,Erlotinib,EGFR,EGFR,0.020,0.058


In [17]:
%%bigquery
-- SQL equivalent: self-join on interactions
SELECT
  c1.compound_name         AS compound_a,
  c2.compound_name         AS compound_b,
  t.target_name            AS shared_target,
  t.gene_name,
  ROUND(i1.affinity_nm, 3) AS affinity_a_nm,
  ROUND(i2.affinity_nm, 3) AS affinity_b_nm
FROM `bqml-demo-sudipto.drug_target_graph.interactions` i1
JOIN `bqml-demo-sudipto.drug_target_graph.interactions` i2
  ON i1.target_id = i2.target_id AND i1.compound_id < i2.compound_id
JOIN `bqml-demo-sudipto.drug_target_graph.compounds` c1 ON i1.compound_id = c1.compound_id
JOIN `bqml-demo-sudipto.drug_target_graph.compounds` c2 ON i2.compound_id = c2.compound_id
JOIN `bqml-demo-sudipto.drug_target_graph.targets`   t  ON i1.target_id   = t.target_id
WHERE c1.therapeutic_area = 'Oncology'
  AND c2.therapeutic_area = 'Oncology'
ORDER BY c1.compound_name, c2.compound_name;
-- Self-join on interactions is error-prone; GQL expresses it naturally with <-[b2]-(c2)

Query is running:   0%|          |

Downloading:   0%|          |

,compound_a,compound_b,shared_target,gene_name,affinity_a_nm,affinity_b_nm
0,Compound-X1,Compound-X3,hERG,KCNH2,1.200,0.950
1,Crizotinib,Compound-X3,VEGFR2,KDR,0.410,0.120
2,Erlotinib,Compound-X1,EGFR,EGFR,0.058,0.041
3,Erlotinib,Compound-X1,hERG,KCNH2,8.500,1.200
4,Erlotinib,Compound-X3,hERG,KCNH2,8.500,0.950
5,Erlotinib,Sorafenib,hERG,KCNH2,8.500,2.800
6,Gefitinib,Compound-X1,EGFR,EGFR,0.020,0.041
7,Gefitinib,Compound-X1,hERG,KCNH2,3.100,1.200
8,Gefitinib,Compound-X3,hERG,KCNH2,3.100,0.950
9,Gefitinib,Erlotinib,EGFR,EGFR,0.020,0.058


In [18]:
%%bigquery --graph display_only
-- Visualise: shared target network between oncology compounds
GRAPH `bqml-demo-sudipto.drug_target_graph.drug_target_interaction_graph`
MATCH (c1:Compound)-[b1:BINDS_TO]->(t:Target)<-[b2:BINDS_TO]-(c2:Compound)
WHERE c1.compound_id < c2.compound_id
  AND c1.therapeutic_area = 'Oncology'
RETURN
  TO_JSON(c1) AS compound_a,
  TO_JSON(b1) AS binds_to_a,
  TO_JSON(t)  AS shared_target,
  TO_JSON(b2) AS binds_to_b,
  TO_JSON(c2) AS compound_b

Query is running:   0%|          |

Downloading:   0%|          |

---
## Q4: Full disease pathway coverage per compound
*Trace every compound through its targets to all affected pathways and disease areas — the therapeutic blast radius*

In [19]:
%%bigquery
-- GQL: 2-hop traversal aggregated by disease area
SELECT
  compound_name,
  dev_stage,
  disease_area,
  COUNT(DISTINCT pathway_id) AS pathways_affected,
  COUNT(DISTINCT target_id)  AS targets_involved,
  STRING_AGG(DISTINCT pathway_name ORDER BY pathway_name) AS pathway_list
FROM GRAPH_TABLE(
  `bqml-demo-sudipto.drug_target_graph.drug_target_interaction_graph`
  MATCH (c:Compound)-[b:BINDS_TO]->(t:Target)-[p:PARTICIPATES_IN]->(pw:Pathway)
  COLUMNS (
    c.compound_name AS compound_name,
    c.dev_stage     AS dev_stage,
    t.target_id     AS target_id,
    pw.pathway_id   AS pathway_id,
    pw.pathway_name AS pathway_name,
    pw.disease_area AS disease_area
  )
)
GROUP BY compound_name, dev_stage, disease_area
ORDER BY compound_name, pathways_affected DESC;

Query is running:   0%|          |

Downloading:   0%|          |

,compound_name,dev_stage,disease_area,pathways_affected,targets_involved,pathway_list
0,Compound-X1,Phase II,Oncology,4,2,"ErbB Signaling,MAPK Signaling,PI3K-AKT Signali..."
1,Compound-X1,Phase II,Cardiovascular,2,1,"Cardiac Muscle Contraction,hERG Cardiac Channel"
2,Compound-X2,Phase I,Oncology,2,2,"PI3K-AKT Signaling,mTOR Signaling"
3,Compound-X3,Preclinical,Oncology,3,3,"MAPK Signaling,RAS Signaling,VEGF Signaling"
4,Compound-X3,Preclinical,Cardiovascular,2,1,"Cardiac Muscle Contraction,hERG Cardiac Channel"
5,Crizotinib,Approved,Oncology,3,2,"ALK Signaling,MAPK Signaling,VEGF Signaling"
6,Erlotinib,Approved,Oncology,3,1,"ErbB Signaling,MAPK Signaling,PI3K-AKT Signaling"
7,Erlotinib,Approved,Cardiovascular,2,1,"Cardiac Muscle Contraction,hERG Cardiac Channel"
8,Gefitinib,Approved,Oncology,3,1,"ErbB Signaling,MAPK Signaling,PI3K-AKT Signaling"
9,Gefitinib,Approved,Cardiovascular,2,1,"Cardiac Muscle Contraction,hERG Cardiac Channel"


In [20]:
%%bigquery
-- SQL equivalent
SELECT
  c.compound_name,
  c.dev_stage,
  pw.disease_area,
  COUNT(DISTINCT pw.pathway_id) AS pathways_affected,
  COUNT(DISTINCT t.target_id)   AS targets_involved,
  STRING_AGG(DISTINCT pw.pathway_name ORDER BY pw.pathway_name) AS pathway_list
FROM `bqml-demo-sudipto.drug_target_graph.interactions`    i
JOIN `bqml-demo-sudipto.drug_target_graph.compounds`       c  ON i.compound_id = c.compound_id
JOIN `bqml-demo-sudipto.drug_target_graph.targets`         t  ON i.target_id   = t.target_id
JOIN `bqml-demo-sudipto.drug_target_graph.target_pathways` tp ON t.target_id   = tp.target_id
JOIN `bqml-demo-sudipto.drug_target_graph.pathways`        pw ON tp.pathway_id = pw.pathway_id
GROUP BY c.compound_name, c.dev_stage, pw.disease_area
ORDER BY c.compound_name, pathways_affected DESC;

Query is running:   0%|          |

Downloading:   0%|          |

,compound_name,dev_stage,disease_area,pathways_affected,targets_involved,pathway_list
0,Compound-X1,Phase II,Oncology,4,2,"ErbB Signaling,MAPK Signaling,PI3K-AKT Signali..."
1,Compound-X1,Phase II,Cardiovascular,2,1,"Cardiac Muscle Contraction,hERG Cardiac Channel"
2,Compound-X2,Phase I,Oncology,2,2,"PI3K-AKT Signaling,mTOR Signaling"
3,Compound-X3,Preclinical,Oncology,3,3,"MAPK Signaling,RAS Signaling,VEGF Signaling"
4,Compound-X3,Preclinical,Cardiovascular,2,1,"Cardiac Muscle Contraction,hERG Cardiac Channel"
5,Crizotinib,Approved,Oncology,3,2,"ALK Signaling,MAPK Signaling,VEGF Signaling"
6,Erlotinib,Approved,Oncology,3,1,"ErbB Signaling,MAPK Signaling,PI3K-AKT Signaling"
7,Erlotinib,Approved,Cardiovascular,2,1,"Cardiac Muscle Contraction,hERG Cardiac Channel"
8,Gefitinib,Approved,Oncology,3,1,"ErbB Signaling,MAPK Signaling,PI3K-AKT Signaling"
9,Gefitinib,Approved,Cardiovascular,2,1,"Cardiac Muscle Contraction,hERG Cardiac Channel"


In [21]:
%%bigquery --graph display_only
-- Visualise: full pipeline compound graph
GRAPH `bqml-demo-sudipto.drug_target_graph.drug_target_interaction_graph`
MATCH (c:Compound)-[b:BINDS_TO]->(t:Target)-[p:PARTICIPATES_IN]->(pw:Pathway)
WHERE c.dev_stage IN ('Phase I', 'Phase II', 'Preclinical')
RETURN
  TO_JSON(c)  AS compound,
  TO_JSON(b)  AS binds_to,
  TO_JSON(t)  AS target,
  TO_JSON(p)  AS participates_in,
  TO_JSON(pw) AS pathway

Query is running:   0%|          |

Downloading:   0%|          |

---
## Q5: Which pipeline compounds hit oncology targets but avoid hERG?
*Safety-first compound selection: good oncology coverage + clean cardiac profile*

In [22]:
%%bigquery
-- GQL
WITH oncology_compounds AS (
  SELECT DISTINCT compound_id, compound_name, dev_stage
  FROM GRAPH_TABLE(
    `bqml-demo-sudipto.drug_target_graph.drug_target_interaction_graph`
    MATCH (c:Compound)-[b:BINDS_TO]->(t:Target)-[p:PARTICIPATES_IN]->(pw:Pathway)
    WHERE pw.disease_area     = 'Oncology'
      AND pw.cancer_relevance = 'high'
    COLUMNS (
      c.compound_id   AS compound_id,
      c.compound_name AS compound_name,
      c.dev_stage     AS dev_stage
    )
  )
),
herg_risk_compounds AS (
  SELECT DISTINCT compound_id
  FROM GRAPH_TABLE(
    `bqml-demo-sudipto.drug_target_graph.drug_target_interaction_graph`
    MATCH (c:Compound)-[b:BINDS_TO]->(t:Target)
    WHERE t.gene_name = 'KCNH2'
    COLUMNS (c.compound_id AS compound_id)
  )
)
SELECT
  o.compound_name,
  o.dev_stage,
  'Clean cardiac profile' AS cardiac_safety
FROM oncology_compounds o
LEFT JOIN herg_risk_compounds h ON o.compound_id = h.compound_id
WHERE h.compound_id IS NULL
ORDER BY o.dev_stage, o.compound_name;

Query is running:   0%|          |

Downloading:   0%|          |

,compound_name,dev_stage,cardiac_safety
0,Crizotinib,Approved,Clean cardiac profile
1,Idelalisib,Approved,Clean cardiac profile
2,Palbociclib,Approved,Clean cardiac profile
3,Rapamycin,Approved,Clean cardiac profile
4,Trametinib,Approved,Clean cardiac profile
5,Vemurafenib,Approved,Clean cardiac profile
6,Compound-X2,Phase I,Clean cardiac profile


In [23]:
%%bigquery
-- SQL equivalent
WITH oncology_compounds AS (
  SELECT DISTINCT c.compound_id, c.compound_name, c.dev_stage
  FROM `bqml-demo-sudipto.drug_target_graph.interactions`    i
  JOIN `bqml-demo-sudipto.drug_target_graph.compounds`       c  ON i.compound_id = c.compound_id
  JOIN `bqml-demo-sudipto.drug_target_graph.targets`         t  ON i.target_id   = t.target_id
  JOIN `bqml-demo-sudipto.drug_target_graph.target_pathways` tp ON t.target_id   = tp.target_id
  JOIN `bqml-demo-sudipto.drug_target_graph.pathways`        pw ON tp.pathway_id = pw.pathway_id
  WHERE pw.disease_area     = 'Oncology'
    AND pw.cancer_relevance = 'high'
),
herg_risk_compounds AS (
  SELECT DISTINCT i.compound_id
  FROM `bqml-demo-sudipto.drug_target_graph.interactions` i
  JOIN `bqml-demo-sudipto.drug_target_graph.targets`      t ON i.target_id = t.target_id
  WHERE t.gene_name = 'KCNH2'
)
SELECT
  o.compound_name,
  o.dev_stage,
  'Clean cardiac profile' AS cardiac_safety
FROM oncology_compounds o
LEFT JOIN herg_risk_compounds h ON o.compound_id = h.compound_id
WHERE h.compound_id IS NULL
ORDER BY o.dev_stage, o.compound_name;
-- Two separate 4-JOIN chains vs two clean GQL MATCH patterns

Query is running:   0%|          |

Downloading:   0%|          |

,compound_name,dev_stage,cardiac_safety
0,Crizotinib,Approved,Clean cardiac profile
1,Idelalisib,Approved,Clean cardiac profile
2,Palbociclib,Approved,Clean cardiac profile
3,Rapamycin,Approved,Clean cardiac profile
4,Trametinib,Approved,Clean cardiac profile
5,Vemurafenib,Approved,Clean cardiac profile
6,Compound-X2,Phase I,Clean cardiac profile


---
## 📊 Summary: SQL vs GQL

| Query | SQL JOINs | GQL pattern | Key SQL pitfall |
|---|---|---|---|
| Q1: Target binding profile | 2 JOINs | 1-hop MATCH | Readable but verbose |
| Q2: hERG cardiac risk | 4 JOINs | 2-hop MATCH | Easy to miss pathway filter |
| Q3: Shared target pairs | 5 JOINs + self-join | Bidirectional 1-hop | Fan-out bugs from double join |
| Q4: Disease pathway coverage | 4 JOINs | 2-hop MATCH | Hard to extend to 3 hops |
| Q5: Safe compound selection | Two 4-JOIN CTEs | Two 1-hop MATCH blocks | Duplicating join logic |

### Core insight

Graph queries express **relationships as first-class citizens**. In drug discovery, where the science is fundamentally about how entities connect — compounds bind targets, targets participate in pathways, pathways drive disease — GQL maps directly to how scientists think. You write queries that read like scientific questions, not database plumbing.

# ⚡ GQL vs SQL — Performance at Scale

---

## The Direct Answer

**GQL itself is not a different execution engine.** In BigQuery, `GRAPH_TABLE` compiles down to the same underlying SQL execution plan — joins, scans, and aggregations — that a well-written SQL query would produce. So at the query execution level, **a GQL query and its SQL equivalent will perform similarly** on the same data.

---

## Where Graph *Does* Give You Real Performance Wins at Scale

The performance advantage of graph in BigQuery (and graph databases generally) shows up in specific scenarios:

### 1. Multi-hop traversal on sparse graphs

If your interaction graph is sparse — meaning each compound binds only a small number of targets out of millions — graph-aware storage and indexing can prune the search space dramatically. A SQL self-join on a 10M-row interactions table has to consider far more row combinations than a graph traversal that follows only the edges that actually exist.

> In BigQuery specifically, this matters less because BQ is a columnar scan engine optimised for full-table analytics. But in a native graph DB like Neo4j, this is the primary performance argument.

### 2. Avoiding intermediate result explosion

The classic SQL problem at scale is **join fan-out** — when you join A→B→C, the intermediate result of A→B can be enormous before you filter it down with the C join. GQL's execution model can in principle filter earlier in the traversal, reducing the size of intermediate results.

### 3. Index-free adjacency (not in BQ, but worth knowing)

Native graph databases store edges as direct pointers between nodes — so traversing 1 million hops doesn't require scanning a table, it just follows pointers. BigQuery doesn't do this, but it's why Neo4j, Amazon Neptune, and TigerGraph can be orders of magnitude faster for deep traversal queries.

---

## What BigQuery's Graph *Actually* Optimises For

BigQuery's property graph implementation is optimised for **analytical graph workloads** — not transactional graph traversal. The real wins are:

| Scenario | BQ Graph advantage |
|---|---|
| Running graph queries on data that already lives in BQ | No ETL to a separate graph DB — massive operational win |
| Combining graph traversal with analytical aggregations | Native SQL aggregations on top of graph patterns in one query |
| Teams without graph DB expertise | GQL is learnable; operating Neo4j at scale is a specialised skill |
| Governance, security, lineage | One platform, one IAM model, one audit log |

---

## The Honest Comparison at Scale

| Workload type | BQ GQL | Dedicated graph DB (Neo4j etc.) |
|---|---|---|
| Deep traversal (10+ hops) | ⚠️ Slow | ✅ Fast (index-free adjacency) |
| Sparse graph, billions of edges | ⚠️ OK | ✅ Fast |
| Analytical aggregations on graph | ✅ Fast | ⚠️ Not their strength |
| Data already in your warehouse | ✅ Win | ❌ Requires ETL |
| Real-time graph queries (<100ms) | ❌ No | ✅ Yes |
| Ad-hoc analysis by data teams | ✅ Yes | ⚠️ Needs Cypher expertise |

---

## The Positioning for Your Demo

The way to frame this to a customer is:

> *"If your primary use case is deep real-time graph traversal at millisecond latency — like a fraud detection system checking 10 hops in under 100ms — you probably need a dedicated graph database. But if your graph data already lives in BigQuery, your team knows SQL, and you want to run graph-shaped analytical queries alongside your existing warehouse workloads — BQ graph gives you 80% of the capability with zero additional infrastructure, zero ETL, and no new database to operate."*

The strongest argument isn't raw query speed — it's **eliminating the operational overhead of running a separate graph database** while still being able to express and answer graph-shaped questions naturally.